In [ ]:
"""
eval.py  —  BAGIAN 3 (REVISI): evaluasi fair, multi-metrik, per-bahasa,
            + bootstrap CI, decoding seragam HF/GGUF, metrik overall yang jujur.
 
Mengevaluasi SATU model pada test_final.jsonl dan menyimpan hasil JSON. Untuk
membuktikan "peningkatan" (RQ1), jalankan untuk KEEMPAT konfigurasi dgn protokol
identik, lalu ringkas deltanya + signifikansinya:
 
    python eval.py --model unsloth/Qwen3.5-2B                --model_type qwen  --label qwen_baseline
    python eval.py --model outputs/merged/qwen35-2b-medical  --model_type qwen  --label qwen_finetuned
    python eval.py --model unsloth/gemma-4-E2B-it            --model_type gemma --label gemma_baseline
    python eval.py --model outputs/merged/gemma4-e2b-medical --model_type gemma --label gemma_finetuned
    python eval.py --summarize results                       # tabel + delta + CI signifikansi
 
Protokol FAIR (3.4): test_final SAMA, n_eval SAMA, seed SAMA (urutan sampel identik
antar run -> memungkinkan PAIRED bootstrap), greedy (do_sample=False),
max_new_tokens=256, format prompt SAMA (chat_format.py).
 
DECODING SERAGAM (perbaikan fairness Q4 vs 16-bit): kedua backend (HF & GGUF)
memakai greedy + repetition_penalty yang SAMA. no_repeat_ngram_size default OFF
karena tak bisa direplikasi di llama.cpp; bila dinyalakan (>0) hanya berlaku di HF
dan akan diberi peringatan (jangan dipakai untuk perbandingan Q4 vs 16-bit).
 
Metrik (3.2):
  - MCQA berhuruf (medmcqa)      -> Exact Match huruf opsi (accuracy)
  - Yes/No/Maybe (pubmedqa,...)  -> Exact Match ternormalisasi (accuracy)
  - Open-ended (sisanya)         -> ROUGE-L, ROUGE-1, token-F1
Dipecah per BAHASA (ID vs EN) (3.3) dan per bucket. Setiap angka utama disertai
bootstrap 95% CI. "overall" TIDAK lagi dilaporkan sebagai satu angka rougeL yang
menyesatkan; diganti kolom yang tiap-tiapnya metrik tunggal yang terdefinisi jelas.
"""
import argparse
import glob
import json
import os
import random
import re
import warnings
from collections import Counter, defaultdict
 
warnings.filterwarnings("ignore")
 
from chat_format import MODEL_MODES, split_prompt_reference, clean_greeting  # noqa: E402
 
# --------------------------------------------------------------------------- #
# Deteksi bahasa (pakai field eksplisit bila ada; heuristik hanya fallback)
# --------------------------------------------------------------------------- #
_ID_SOURCES = ("indonesia_qna", "alodokter", "ppk", "kemenkes", "indonesia", "id_med")
_ID_KW = ["puskesmas", "dokter", "pasien", "obat", "demam", "rumah sakit", "kesehatan",
          "penyakit", "gejala", "keluhan", "saya", "yang", "dan", "tidak", "dengan"]
 
 
def lang_of(sample):
    """Prioritas: field 'lang'/'language' eksplisit -> source -> heuristik kata."""
    lg = str(sample.get("lang") or sample.get("language") or "").lower()
    if lg in ("id", "ind", "indonesia", "indonesian"):
        return "id"
    if lg in ("en", "eng", "english"):
        return "en"
    src = sample.get("source", "").lower()
    if any(x in src for x in _ID_SOURCES):
        return "id"
    c = " ".join(m.get("content", "") for m in sample.get("messages", [])).lower()
    return "id" if sum(w in c for w in _ID_KW) >= 3 else "en"
 
 
# --------------------------------------------------------------------------- #
# Metrik (self-contained, tanpa dependensi eksternal)
# --------------------------------------------------------------------------- #
_TOK = re.compile(r"[a-z0-9]+")
 
 
def toks(s):
    return _TOK.findall(s.lower())
 
 
def _lcs(a, b):
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return 0
    prev = [0] * (m + 1)
    for i in range(1, n + 1):
        cur = [0] * (m + 1)
        ai = a[i - 1]
        for j in range(1, m + 1):
            cur[j] = prev[j - 1] + 1 if ai == b[j - 1] else max(prev[j], cur[j - 1])
        prev = cur
    return prev[m]
 
 
def _f1(common, plen, rlen):
    if common == 0 or plen == 0 or rlen == 0:
        return 0.0
    p, r = common / plen, common / rlen
    return 2 * p * r / (p + r)
 
 
def rouge1(pred, ref):
    pt, rt = toks(pred), toks(ref)
    overlap = sum((Counter(pt) & Counter(rt)).values())
    return _f1(overlap, len(pt), len(rt))
 
 
def rougeL(pred, ref):
    pt, rt = toks(pred), toks(ref)
    return _f1(_lcs(pt, rt), len(pt), len(rt))
 
 
def token_f1(pred, ref):
    pt, rt = toks(pred), toks(ref)
    common = sum((Counter(pt) & Counter(rt)).values())
    return _f1(common, len(pt), len(rt))
 
 
_LETTER = [
    re.compile(r"correct answer is\s*\**\s*\(?\s*([A-E])\b", re.I),
    re.compile(r"answer\s*[:=]?\s*\**\s*\(?\s*([A-E])\b", re.I),
    re.compile(r"^\s*\(?\s*([A-E])[).:]", re.I),
    re.compile(r"\b([A-E])\)", re.I),
]
 
 
def extract_letter(text):
    for pat in _LETTER:
        m = pat.search(text)
        if m:
            return m.group(1).upper()
    return None
 
 
def extract_yesno(text):
    m = re.match(r"\s*\**\s*(yes|no|maybe|ya|tidak|mungkin)\b", text.strip(), re.I)
    if not m:
        return None
    w = m.group(1).lower()
    return {"ya": "yes", "tidak": "no", "mungkin": "maybe"}.get(w, w)
 
 
def bucket_of(sample, ref):
    src = sample.get("source", "")
    user = " ".join(m["content"] for m in sample["messages"] if m["role"] == "user")
    if src == "medmcqa" or re.search(r"\n\s*[A-E]\)", user):
        return "mcqa"
    if src == "pubmedqa" or extract_yesno(ref) in ("yes", "no", "maybe"):
        return "yesno"
    return "open"
 
 
# --------------------------------------------------------------------------- #
# Bootstrap CI (perbaikan #1: signifikansi)
# --------------------------------------------------------------------------- #
def bootstrap_ci(values, n_boot, seed, alpha=0.05):
    """CI 95% rata-rata via percentile bootstrap. Kembalikan (mean, lo, hi, n)."""
    n = len(values)
    if n == 0:
        return None
    mean = sum(values) / n
    if n_boot <= 0 or n == 1:
        return {"mean": round(mean, 4), "lo": round(mean, 4), "hi": round(mean, 4), "n": n}
    rng = random.Random(seed)
    means = []
    for _ in range(n_boot):
        s = 0.0
        for _ in range(n):
            s += values[rng.randrange(n)]
        means.append(s / n)
    means.sort()
    lo = means[max(0, int((alpha / 2) * n_boot))]
    hi = means[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]
    return {"mean": round(mean, 4), "lo": round(lo, 4), "hi": round(hi, 4), "n": n}
 
 
def paired_delta_ci(a, b, n_boot, seed, alpha=0.05):
    """PAIRED bootstrap selisih (a-b) pada sampel yang sama (urutan identik antar run).
    Signifikan jika CI tidak memuat 0. Kembalikan dict + flag 'sig'."""
    n = min(len(a), len(b))
    if n == 0:
        return None
    diffs = [a[i] - b[i] for i in range(n)]
    mean = sum(diffs) / n
    if n_boot <= 0 or n == 1:
        return {"delta": round(mean, 4), "lo": round(mean, 4), "hi": round(mean, 4),
                "n": n, "sig": False}
    rng = random.Random(seed)
    means = []
    for _ in range(n_boot):
        s = 0.0
        for _ in range(n):
            s += diffs[rng.randrange(n)]
        means.append(s / n)
    means.sort()
    lo = means[max(0, int((alpha / 2) * n_boot))]
    hi = means[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]
    return {"delta": round(mean, 4), "lo": round(lo, 4), "hi": round(hi, 4),
            "n": n, "sig": (lo > 0 or hi < 0)}
 
 
# --------------------------------------------------------------------------- #
# Loading model
# --------------------------------------------------------------------------- #
def load_model(args):
    """Kembalikan (kind, handle, tokenizer). kind in {'hf','gguf'}."""
    from transformers import AutoTokenizer
    if args.gguf:
        from llama_cpp import Llama
        llm = Llama(model_path=args.gguf, n_ctx=4096, n_gpu_layers=-1,
                    n_threads=os.cpu_count(), verbose=False, logits_all=False)
        tok = AutoTokenizer.from_pretrained(args.model)  # merged dir -> hanya utk template
        return "gguf", llm, tok
 
    if args.loader == "unsloth":
        import unsloth
        Loader = unsloth.FastVisionModel
        try:
            model, tok = Loader.from_pretrained(
                model_name=args.model, max_seq_length=args.max_seq_length,
                load_in_4bit=args.load_in_4bit, dtype=None)
        except TypeError:
            model, tok = Loader.from_pretrained(
                model_name=args.model, load_in_4bit=args.load_in_4bit, dtype=None)
        Loader.for_inference(model)
        tok = getattr(tok, "tokenizer", tok)
    else:
        import torch
        from transformers import AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(args.model)
        model = AutoModelForCausalLM.from_pretrained(
            args.model, torch_dtype=torch.bfloat16, device_map="auto")
        model.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return "hf", model, tok
 
 
# --------------------------------------------------------------------------- #
# Generate (greedy + repetition_penalty SAMA di kedua backend -> fair)
# --------------------------------------------------------------------------- #
def generate_all(kind, handle, tok, prompts, args):
    preds = []
    if kind == "gguf":
        for i, p in enumerate(prompts):
            o = handle.create_completion(
                p, max_tokens=args.max_new_tokens, temperature=0.0, top_k=1,
                repeat_penalty=args.repetition_penalty)  # greedy, penalti identik HF
            preds.append(o["choices"][0]["text"])
            print(f"  generated {i + 1}/{len(prompts)}", end="\r")
        print()
        return preds
 
    import torch
    dev = next(handle.parameters()).device
    gen_kwargs = dict(max_new_tokens=args.max_new_tokens, do_sample=False, num_beams=1,
                      repetition_penalty=args.repetition_penalty,
                      pad_token_id=tok.pad_token_id)
    if args.no_repeat_ngram_size > 0:  # ablasi HF-only; jangan utk Q4 vs 16-bit
        gen_kwargs["no_repeat_ngram_size"] = args.no_repeat_ngram_size
    for i in range(0, len(prompts), args.batch_size):
        batch = prompts[i:i + args.batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(dev)
        with torch.no_grad():
            out = handle.generate(**enc, **gen_kwargs)
        gen = out[:, enc["input_ids"].shape[1]:]
        preds.extend(tok.batch_decode(gen, skip_special_tokens=True))
        print(f"  generated {min(i + args.batch_size, len(prompts))}/{len(prompts)}", end="\r")
    print()
    return preds
 
 
# --------------------------------------------------------------------------- #
# Evaluasi satu model
# --------------------------------------------------------------------------- #
def evaluate(args):
    mode = MODEL_MODES[args.model_type]
 
    samples = [json.loads(l) for l in open(args.test_file, encoding="utf-8")]
    random.seed(args.seed)
    random.shuffle(samples)
    samples = samples[:args.n_eval]
    print(f"[{args.label}] model={args.model} n={len(samples)}")
 
    if args.gguf and args.no_repeat_ngram_size > 0:
        print("  [WARN] no_repeat_ngram_size diabaikan di GGUF; jangan pakai >0 untuk "
              "perbandingan Q4 vs 16-bit (decoding jadi tak setara).")
 
    kind, handle, tok = load_model(args)
 
    prompts, refs, buckets, langs = [], [], [], []
    for s in samples:
        p, r = split_prompt_reference(s, mode, tok)
        prompts.append(p)
        refs.append(r)
        buckets.append(bucket_of(s, r))
        langs.append(lang_of(s))
 
    preds = generate_all(kind, handle, tok, prompts, args)
 
    if not args.no_postclean:
        preds = [clean_greeting(p) for p in preds]
 
    # Perbaikan #5: validasi parse referensi MCQA -> jangan diam2 dihitung salah
    mcqa_total = sum(1 for b in buckets if b == "mcqa")
    mcqa_ref_unparsed = sum(1 for b, r in zip(buckets, refs)
                            if b == "mcqa" and extract_letter(r) is None)
    if mcqa_total and mcqa_ref_unparsed:
        pct = 100 * mcqa_ref_unparsed / mcqa_total
        print(f"  [WARN] {mcqa_ref_unparsed}/{mcqa_total} ({pct:.1f}%) referensi MCQA "
              f"tak ter-parse hurufnya -> otomatis dihitung SALAH. Periksa format ref.")
 
    # akumulasi metrik per kolom + simpan SKOR PER-SAMPEL (untuk paired bootstrap antar run)
    agg = defaultdict(lambda: defaultdict(list))
    per_sample = []          # urutan identik antar run (seed sama) -> bisa di-pair
    examples = []
    for pred, ref, b, lg in zip(preds, refs, buckets, langs):
        # kolom: bucket spesifik, closed gabungan, open per-bahasa
        if b in ("mcqa", "yesno"):
            if b == "mcqa":
                hit = float(extract_letter(pred) is not None and
                            extract_letter(pred) == extract_letter(ref))
            else:
                gp, gr = extract_yesno(pred), extract_yesno(ref)
                hit = float(gp is not None and gp == gr)
            for k in (f"bucket:{b}", "closed", f"closed|lang:{lg}"):
                agg[k]["accuracy"].append(hit)
            primary = ("accuracy", hit)
        else:
            rl, r1, f1 = rougeL(pred, ref), rouge1(pred, ref), token_f1(pred, ref)
            for k in ("bucket:open", f"bucket:open|lang:{lg}"):
                agg[k]["rougeL"].append(rl)
                agg[k]["rouge1"].append(r1)
                agg[k]["token_f1"].append(f1)
            primary = ("rougeL", rl)
        per_sample.append({"bucket": b, "lang": lg,
                           "metric": primary[0], "score": primary[1]})
        if len(examples) < 12:
            examples.append({"bucket": b, "lang": lg, "pred": pred[:300], "ref": ref[:300]})
 
    # ringkas tiap kolom: semua metrik + CI bootstrap utk metrik utamanya
    def primary_metric(key):
        return "accuracy" if (key.startswith("bucket:mcqa")
                              or key.startswith("bucket:yesno")
                              or key.startswith("closed")) else "rougeL"
 
    results = {}
    for k, d in sorted(agg.items()):
        entry = {m: round(sum(v) / len(v), 4) for m, v in d.items() if v}
        entry.update({m + "_n": len(v) for m, v in d.items() if v})
        pm = primary_metric(k)
        if pm in d and d[pm]:
            entry["ci"] = bootstrap_ci(d[pm], args.n_bootstrap, args.seed)
        results[k] = entry
 
    # peringatan sel kecil (perbaikan #4)
    for k, e in results.items():
        ci = e.get("ci")
        if ci and ci["n"] < args.min_cell_n:
            print(f"  [WARN] kolom '{k}' n={ci['n']} (< {args.min_cell_n}) -> "
                  f"estimasi tidak stabil, CI lebar.")
 
    out = {
        "label": args.label,
        "model": args.model,
        "model_type": args.model_type,
        "config": {
            "n_eval": len(samples), "max_new_tokens": args.max_new_tokens,
            "do_sample": False, "repetition_penalty": args.repetition_penalty,
            "no_repeat_ngram_size": args.no_repeat_ngram_size,
            "batch_size": args.batch_size, "n_bootstrap": args.n_bootstrap,
            "quant": "Q4_K_M(gguf)" if args.gguf else ("4bit" if args.load_in_4bit else "16bit"),
            "loader": "gguf" if args.gguf else args.loader, "gguf": args.gguf,
            "test_file": args.test_file, "seed": args.seed,
            "postclean": not args.no_postclean,
            "mcqa_ref_unparsed": mcqa_ref_unparsed, "mcqa_total": mcqa_total,
        },
        "metrics": results,
        "per_sample": per_sample,   # dipakai summarize utk paired bootstrap
        "examples": examples,
    }
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    with open(args.out, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\nSaved -> {args.out}")
 
    # Perbaikan #6: dump SEMUA pred+ref utk review kualitatif klinis manual
    if not args.no_dump:
        dump_path = os.path.join(os.path.dirname(args.out) or ".",
                                 f"preds_{args.label}.jsonl")
        with open(dump_path, "w", encoding="utf-8") as f:
            for pred, ref, b, lg in zip(preds, refs, buckets, langs):
                f.write(json.dumps({"bucket": b, "lang": lg, "pred": pred, "ref": ref},
                                   ensure_ascii=False) + "\n")
        print(f"Dump pred lengkap -> {dump_path}")
 
    _print_one(out)
 
 
# kolom tampilan: tiap kolom = SATU metrik yang terdefinisi jelas (tak ada 'overall' campuran)
_COLS = [
    ("closed(acc)", "closed", "accuracy"),
    ("mcqa(acc)", "bucket:mcqa", "accuracy"),
    ("yesno(acc)", "bucket:yesno", "accuracy"),
    ("open(RL)", "bucket:open", "rougeL"),
    ("open-id(RL)", "bucket:open|lang:id", "rougeL"),
    ("open-en(RL)", "bucket:open|lang:en", "rougeL"),
]
 
 
def _cell(metrics, key, metric):
    return metrics.get(key, {}).get(metric)
 
 
def _print_one(out):
    m = out["metrics"]
    print(f"\n=== {out['label']} ===")
    for name, key, metric in _COLS:
        v = _cell(m, key, metric)
        ci = m.get(key, {}).get("ci")
        if isinstance(v, (int, float)):
            ci_s = f"  [{ci['lo']:.3f},{ci['hi']:.3f}] n={ci['n']}" if ci else ""
            print(f"  {name:14s} {v:.4f}{ci_s}")
 
 
# --------------------------------------------------------------------------- #
# Ringkasan banyak hasil -> tabel + delta + signifikansi (paired bootstrap)
# --------------------------------------------------------------------------- #
def _scores_for_col(run, key, metric):
    """Ambil skor per-sampel untuk satu kolom (urutan asli) dari per_sample."""
    ps = run.get("per_sample")
    if ps is None:
        return None
    if key == "closed":
        want = lambda r: r["bucket"] in ("mcqa", "yesno")
    elif key == "bucket:mcqa":
        want = lambda r: r["bucket"] == "mcqa"
    elif key == "bucket:yesno":
        want = lambda r: r["bucket"] == "yesno"
    elif key == "bucket:open":
        want = lambda r: r["bucket"] == "open"
    elif key == "bucket:open|lang:id":
        want = lambda r: r["bucket"] == "open" and r["lang"] == "id"
    elif key == "bucket:open|lang:en":
        want = lambda r: r["bucket"] == "open" and r["lang"] == "en"
    else:
        return None
    # urutan dipertahankan -> pairing antar run valid (seed & n sama)
    return [r["score"] for r in ps if want(r)]
 
 
def summarize_results(folder, n_bootstrap, seed):
    files = sorted(glob.glob(os.path.join(folder, "*.json")))
    runs = {}
    for f in files:
        if os.path.basename(f).startswith("preds_"):
            continue
        d = json.load(open(f, encoding="utf-8"))
        if "label" in d and "metrics" in d:
            runs[d["label"]] = d
    if not runs:
        print("Tak ada hasil JSON di", folder)
        return
 
    head = f"{'label':22s} | " + " | ".join(f"{n:>11s}" for n, _, _ in _COLS)
    print("\n" + head)
    print("-" * len(head))
    for label, d in runs.items():
        cells = []
        for _, key, metric in _COLS:
            v = _cell(d["metrics"], key, metric)
            cells.append(f"{v:11.4f}" if isinstance(v, (int, float)) else f"{'-':>11s}")
        print(f"{label:22s} | " + " | ".join(cells))
 
    print("\n--- PENINGKATAN (finetuned - baseline), * = signifikan (CI95 tak memuat 0) ---")
    for ft in [l for l in runs if l.endswith("_finetuned")]:
        base = ft.replace("_finetuned", "_baseline")
        if base not in runs:
            continue
        cells = []
        for _, key, metric in _COLS:
            a = _scores_for_col(runs[ft], key, metric)
            b = _scores_for_col(runs[base], key, metric)
            if a and b:
                dci = paired_delta_ci(a, b, n_bootstrap, seed)
                star = "*" if dci["sig"] else " "
                cells.append(f"{dci['delta']:+10.4f}{star}")
            else:
                # fallback ke selisih mean bila per_sample tak tersedia (JSON lama)
                av = _cell(runs[ft]["metrics"], key, metric)
                bv = _cell(runs[base]["metrics"], key, metric)
                cells.append(f"{av - bv:+10.4f} " if isinstance(av, (int, float))
                             and isinstance(bv, (int, float)) else f"{'-':>11s}")
        print(f"{ft.replace('_finetuned',''):22s} | " + " | ".join(cells))
    print("\nCatatan: 'closed' = MCQA+Yes/No digabung (accuracy); 'open' = ROUGE-L. "
          "Tidak ada angka 'overall' tunggal karena metriknya heterogen.")
 
 
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--summarize", metavar="FOLDER", help="ringkas semua *.json di folder")
    ap.add_argument("--model", help="path model merged ATAU HF id (baseline)")
    ap.add_argument("--model_type", choices=["qwen", "gemma"])
    ap.add_argument("--label", default="run")
    ap.add_argument("--test_file", default="Data/processed_final/test_final.jsonl")
    ap.add_argument("--n_eval", type=int, default=500,
                    help="naikkan bila sel ID/open terlalu kecil (default 500)")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_new_tokens", type=int, default=256)
    ap.add_argument("--max_seq_length", type=int, default=2048)
    ap.add_argument("--loader", choices=["unsloth", "hf"], default="unsloth")
    ap.add_argument("--load_in_4bit", action="store_true",
                    help="eval pada bobot 4-bit (default 16-bit utk Bagian 3)")
    ap.add_argument("--gguf", default=None,
                    help="path file .gguf (Q4_K_M) -> eval terkuantisasi (Bagian 4)")
    ap.add_argument("--repetition_penalty", type=float, default=1.1,
                    help="SAMA di HF & GGUF -> decoding fair (1.0 = murni greedy)")
    ap.add_argument("--no_repeat_ngram_size", type=int, default=0,
                    help="HF-only, ablasi. JANGAN >0 utk perbandingan Q4 vs 16-bit")
    ap.add_argument("--n_bootstrap", type=int, default=2000,
                    help="iterasi bootstrap utk CI (0 = nonaktif)")
    ap.add_argument("--min_cell_n", type=int, default=30,
                    help="ambang peringatan sel terlalu kecil")
    ap.add_argument("--out", default=None)
    ap.add_argument("--no_postclean", action="store_true",
                    help="matikan post-processing sapaan pd output (Task 4) -> ablasi")
    ap.add_argument("--no_dump", action="store_true",
                    help="jangan dump preds_*.jsonl (default: dump untuk review klinis)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()
 
    if args.summarize:
        summarize_results(args.summarize, args.n_bootstrap, args.seed)
        return
    assert args.model and args.model_type, "--model dan --model_type wajib"
    if args.out is None:
        args.out = f"results/eval_{args.label}.json"
    evaluate(args)
 
 
if __name__ == "__main__":
    main()